# Задание 1 (2 балла)


Выберите 5 языков в википедии (не тех, что использовались в семинаре). Скачайте по 10 случайных статей для каждого языка. Предобработайте тексты, удаляя лишние теги/отступы/разделители (если они есть). Разделите тексты на предложения и создайте датасет, в котором каждому предложению соответствует язык. Кластеризуйте тексты, используя эбмединг модель из прошлого семинара и любой алгоритм кластеризации. Проверьте качество кластеризации с помощь метрики ARI. Отдельно проанализируйте 3 ошибочно кластеризованных текста (если такие есть).

In [1]:
import os
import re
import json
import time
import warnings
from collections import Counter

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import pandas as pd
import requests
import mwparserfromhell
import stanza
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, confusion_matrix

In [2]:
# скачиваем статьи

HEADERS = {'User-Agent': 'HSE-NLP-course/0.1 (your_email@example.com)'}

def wiki_api(lang, retries=5, **params):
    """Запрос к MediaWiki API с обработкой 429"""
    params = {'action': 'query', 'format': 'json', 'formatversion': 2, **params}
    for attempt in range(retries):
        r = requests.get(f'https://{lang}.wikipedia.org/w/api.php',
                         params=params, headers=HEADERS, timeout=30)
        if r.status_code == 429:
            wait = int(r.headers.get('Retry-After', 2 ** attempt))
            print(f'  429, ждём {wait} с')
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

def get_random_titles(lang, n=10):
    """n случайных заголовков статей."""
    data = wiki_api(lang, list='random', rnnamespace=0, rnlimit=n)
    return [p['title'] for p in data['query']['random']]

def get_plain_text(lang, title):
    """Текст статьи без вики-разметки."""
    data = wiki_api(lang, prop='extracts', explaintext=1,
                    titles=title, redirects=1)
    pages = data['query']['pages']
    if not pages:
        return ''
    return pages[0].get('extract', '') or ''

# предобработка

def clean_wikitext(text):
    """Убираем вики-разметку и лишние символы."""
    if not text:
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\{\{.*?\}\}', ' ', text, flags=re.S)
    text = re.sub(r'\[\[.*?\]\]', ' ', text, flags=re.S)
    text = re.sub(r'==+.*?==+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [3]:
# разбиение на предложения

_stanza_pipelines = {}

def get_stanza_pipeline(lang):
    """Кешируем пайплайны."""
    if lang not in _stanza_pipelines:
        try:
            stanza.download(lang, processors='tokenize', verbose=False)
            _stanza_pipelines[lang] = stanza.Pipeline(
                lang, processors='tokenize', verbose=False
            )
        except Exception as e:
            print(f'  Stanza не поддерживает {lang}: {e}')
            _stanza_pipelines[lang] = None
    return _stanza_pipelines[lang]

def split_sentences(texts, lang, min_words=5):
    """Разбиение на предложения. Если Stanza недоступна — regex."""
    nlp = get_stanza_pipeline(lang)
    sents = []
    for t in texts:
        if not t:
            continue
        if nlp is not None:
            doc = nlp(t)
            parts = [s.text for s in doc.sentences]
        else:
            parts = re.split(r'(?<=[.!?…。！？])\s+', t)
        for s in parts:
            s = s.strip()
            if len(s.split()) >= min_words:
                sents.append(s)
    return sents

In [4]:
# сбор датасета

LANGS = ['ja', 'ar', 'fi', 'ko', 'tr']
LANG_NAMES = {
    'ja': 'японский', 'ar': 'арабский', 'fi': 'финский',
    'ko': 'корейский', 'tr': 'турецкий',
}
N_ARTICLES = 10
CACHE_FILE = 'wiki_texts_task.json'

if os.path.exists(CACHE_FILE):
    print(f'Загружаем кеш из {CACHE_FILE}')
    wiki_texts = json.load(open(CACHE_FILE, encoding='utf-8'))
else:
    wiki_texts = {}
    for lang in LANGS:
        print(f'Скачиваем статьи ({lang})...')
        try:
            titles = get_random_titles(lang, N_ARTICLES)
        except Exception as e:
            print(f'  ошибка получения заголовков: {e}')
            wiki_texts[lang] = []
            continue
        articles = []
        for t in titles:
            try:
                articles.append(get_plain_text(lang, t))
            except Exception as e:
                print(f'  пропускаем {t}: {e}')
            time.sleep(0.4)
        wiki_texts[lang] = articles
        print(f'  {lang}: {len(articles)} статей')
    json.dump(wiki_texts, open(CACHE_FILE, 'w', encoding='utf-8'),
              ensure_ascii=False)

rows = []
for lang, articles in wiki_texts.items():
    for a in articles:
        cleaned = clean_wikitext(a)
        for s in split_sentences([cleaned], lang):
            rows.append({'text': s, 'lang': lang})

df = pd.DataFrame(rows)
print('\nРазмер датасета:', len(df))
print(df.lang.value_counts().to_string())

if len(df) == 0:
    raise SystemExit('Датасет пуст — проверьте скачивание и Stanza.')

# эмбеддинги

print('\nЗагружаем эмбеддинг-модель...')
model = SentenceTransformer('intfloat/multilingual-e5-base')

def embed(texts, prefix='passage: '):
    """E5 требует префиксы query:/passage:."""
    return model.encode(
        [prefix + t for t in texts],
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=16,
    )

embeddings = embed(df.text.tolist())
print('Размерность эмбеддингов:', embeddings.shape)

# кластеризация и ARI

labels = KMeans(n_clusters=len(LANGS),
                random_state=42, n_init=10).fit_predict(embeddings)
ari = adjusted_rand_score(df.lang, labels)
print(f'\nARI = {ari:.3f}')

df['cluster'] = labels

majority = df.groupby('cluster')['lang'].agg(
    lambda x: x.value_counts().index[0]
)
df['pred'] = df['cluster'].map(majority)

langs_sorted = sorted(df.lang.unique())
cm = confusion_matrix(df.lang, df.pred, labels=langs_sorted)
print('\nConfusion matrix (строки — истина, столбцы — предсказание):')
print(pd.DataFrame(cm, index=langs_sorted, columns=langs_sorted))

Загружаем кеш из wiki_texts_task.json

Размер датасета: 240
lang
fi    75
tr    71
ko    52
ja    29
ar    13

Загружаем эмбеддинг-модель...


Batches: 100%|██████████| 15/15 [00:12<00:00,  1.19it/s]


Размерность эмбеддингов: (240, 768)

ARI = 0.786

Confusion matrix (строки — истина, столбцы — предсказание):
    ar  fi  ja  ko  tr
ar   0   0   0  13   0
fi   0  74   0   0   1
ja   0   0  28   0   1
ko   0   0   0  52   0
tr   0   1   0   0  70


In [5]:
# анализ 3 ошибочно кластеризованных текстов

errors = df[df.lang != df.pred].reset_index(drop=True)
print(f'\nОшибок: {len(errors)} из {len(df)} ({len(errors)/len(df):.1%})')

if len(errors) > 0:
    print('\nТри примера ошибочной кластеризации\n')
    for i, row in errors.head(3).iterrows():
        print(f'Пример {i+1}')
        print(f'Истинный язык:  {row.lang} ({LANG_NAMES.get(row.lang, "?")})')
        print(f'Предсказан:     {row.pred} ({LANG_NAMES.get(row.pred, "?")})')
        print(f'Кластер:        {row.cluster}')
        print(f'Текст:          {row.text[:300]}')
        print()

    pair_counts = Counter(zip(errors.lang, errors.pred))
    print('Самые частые путаницы (истина → предсказание):')
    for (true_l, pred_l), cnt in pair_counts.most_common(5):
        print(f'  {true_l} → {pred_l}: {cnt}')
else:
    print('Ошибок нет — все предложения кластеризованы верно.')


Ошибок: 16 из 240 (6.7%)

Три примера ошибочной кластеризации

Пример 1
Истинный язык:  ja (японский)
Предсказан:     tr (турецкий)
Кластер:        4
Текст:          "Boysenberry--A California Treasure" by David Karp Los Angeles Times May 27, 2010:

Пример 2
Истинный язык:  ar (арабский)
Предсказан:     ko (корейский)
Кластер:        3
Текст:          حاشد هي إحدى قرى عزلة القويم بمديرية الطفة التابعة لمحافظة البيضاء، بلغ تعداد سكانها 78 نسمة حسب تعداد اليمن لعام 2004. المركز الوطني للمعلومات باليمن

Пример 3
Истинный язык:  ar (арабский)
Предсказан:     ko (корейский)
Кластер:        3
Текст:          إدوارد إس. زالومون (الإنجليزية: Edward S. Salomon) هو سياسي أمريكي، ولد في 25 ديسمبر 1836 في شلسفيغ هولشتاين في ألمانيا، وتوفي في 18 يوليو 1913 في سان فرانسيسكو في الولايات المتحدة. حزبياً، نشط في الحزب الجمهوري. وقد انتخب Governor of the Territory of Washington ‏ (4 مارس 1870 – أبريل 1872) وانتخب ع

Самые частые путаницы (истина → предсказание):
  ar → ko: 13
  ja → tr: 1
  fi → tr: 1


## Задание 2 (2 балла)
Возьмите какой-нибудь из файлов из Syntagrus или Taiga - 
https://github.com/UniversalDependencies/UD_Russian-SynTagRus/tree/master   
https://tatianashavrina.github.io/taiga_site/downloads  

Используя Dependency Matcher напишите 2 разных query к этому корпусу, результаты которых выдают что-то осмысленное (показывают какую-то закономерность; подвсечивают примеры какого-то феномена; выдают то, что можно использовать в другой задаче). 
Добавьте дополнительные правила фильтрации, если получается слишком зашумленный или наоборот пустой результат (см. семинар и примеры для SVO)
Используйте именно синтактическую структуру, а не просто слова и POS тэги. 
Обязательно покажите примеры, которые находятся по вашим запросам. 


Ответ должен быть структурирован как-то так:
```
## query
Краткое описание того, что вы пытаетесь найти

### код
код с запросом (или несколько ячеек где вы последовательно улучшаете запрос)

### примеры 
несколько самых показательных примеров или даже какая-то агрегация вроде топ-N

... (и так два раза)
```



In [6]:
# Автоустановка spaCy, conllu, ru_core_news_sm
import sys, subprocess

def _pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pkgs])

# spaCy
try:
    import spacy
except ModuleNotFoundError:
    print("Устанавливаю spaCy…")
    _pip_install("spacy")
    import spacy

# conllu
try:
    import conllu
except ModuleNotFoundError:
    print("Устанавливаю conllu…")
    _pip_install("conllu")
    import conllu

# модель
try:
    nlp = spacy.load("ru_core_news_sm")
    print("Модель ru_core_news_sm уже установлена.")
except OSError:
    print("Скачиваю модель ru_core_news_sm…")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "ru_core_news_sm"])
    raise SystemExit("Модель скачана. Перезапустите kernel и выполните ячейку 1 снова.")

print(f"spaCy {spacy.__version__}, модель: {nlp.meta['lang']} {nlp.meta['version']}")

Модель ru_core_news_sm уже установлена.
spaCy 3.8.16, модель: ru 3.8.0


In [7]:
# Чтение CoNLL-U и подготовка Docs
import spacy
import conllu
from spacy.tokens import Doc
from spacy.matcher import DependencyMatcher

nlp = spacy.load("ru_core_news_sm")

CONLLU_PATH = "ru_syntagrus-ud-dev.conllu"   


def read_conllu(path):
    with open(path, encoding="utf-8") as f:
        text = f.read()
    docs = []
    for sent in conllu.parse(text):
        # оставляем только обычные токены (целочисленный id), чтобы multi-word tokens не сдвигали индексы
        tokens = [t for t in sent if isinstance(t["id"], int)]
        doc = Doc(nlp.vocab, words=[t["form"] for t in tokens])
        # zip гарантирует синхронность индексов
        for spacy_tok, conllu_tok in zip(doc, tokens):
            spacy_tok.lemma_ = conllu_tok["lemma"] or ""
            spacy_tok.pos_   = conllu_tok["upos"]  or ""
            spacy_tok.tag_   = conllu_tok["xpos"]  or ""
            spacy_tok.dep_   = conllu_tok["deprel"] or ""    # dep сначала
            h = conllu_tok["head"]
            if isinstance(h, int) and h > 0:
                spacy_tok.head = doc[h - 1]                  # head потом
        docs.append(doc)
    return docs


docs = read_conllu(CONLLU_PATH)
print(f"Загружено предложений: {len(docs)}")
print("Пример:", [t.text for t in docs[0][:10]])

Загружено предложений: 8906
Пример: ['Алгоритм', ',', 'от', 'имени', 'учёного', 'аль', '-', 'Хорезми', ',', '-']


In [8]:
# Тест: глагол с DEP=acl:relcl, у которого head — NOUN
from spacy.matcher import DependencyMatcher

m = DependencyMatcher(nlp.vocab)

pattern_v_acl = [
    {"RIGHT_ID": "v",
     "RIGHT_ATTRS": {"POS": "VERB", "DEP": "acl:relcl"}},
    {"LEFT_ID": "v", "REL_OP": ">",
     "RIGHT_ID": "a",
     "RIGHT_ATTRS": {"POS": {"IN": ["NOUN", "PROPN"]}}},
]

# список паттернов — внешние скобки, внутри один паттерн
m.add("V_ACL", [pattern_v_acl])

hits_v_acl = []
for doc in docs:
    for mid, tokens in m(doc):
        if nlp.vocab.strings[mid] != "V_ACL":
            continue
        hits_v_acl.append([doc[t] for t in tokens])

print(f"V_ACL: {len(hits_v_acl)}")
for ts in hits_v_acl[:5]:
    print("  ", [t.text for t in ts])

V_ACL: 770
   ['определяет', 'последовательность']
   ['приводит', 'числа']
   ['приводит', 'решению']
   ['зависит', 'шаг']
   ['зависит', 'состояния']


In [9]:
# Query 1: относительная клауза «который»
# rel(который, nsubj/obj/obl) <- verb(acl:relcl) -> antecedent(NOUN) -> obj({obj, obl, iobj})
from spacy.matcher import DependencyMatcher

matcher = DependencyMatcher(nlp.vocab)

pattern_rel_subj_obj = [
    # "который" — зависимый от глагола
    {"RIGHT_ID": "rel",
     "RIGHT_ATTRS": {"LEMMA": "который",
                     "DEP": {"IN": ["nsubj", "obj", "obl"]}}},

    # глагол — голова для rel (REL_OP "<": right is head of left)
    {"LEFT_ID": "rel", "REL_OP": "<",
     "RIGHT_ID": "verb",
     "RIGHT_ATTRS": {"POS": "VERB", "DEP": "acl:relcl"}},

    # антецедент — голова глагола
    {"LEFT_ID": "verb", "REL_OP": "<",
     "RIGHT_ID": "antecedent",
     "RIGHT_ATTRS": {"POS": {"IN": ["NOUN", "PROPN"]}}},

    # у глагола есть объект (obj / obl / iobj)
    {"LEFT_ID": "verb", "REL_OP": ">",
     "RIGHT_ID": "obj",
     "RIGHT_ATTRS": {"DEP": {"IN": ["obj", "obl", "iobj"]}}},
]

matcher.add("REL_SUBJ_OBJ", [pattern_rel_subj_obj])

hits_B = []
for doc in docs:
    for mid, tokens in matcher(doc):
        if nlp.vocab.strings[mid] != "REL_SUBJ_OBJ":
            continue
        hits_B.append((doc, [doc[t] for t in tokens]))

print(f"REL_SUBJ_OBJ: {len(hits_B)} совпадений\n")
for doc, ts in hits_B[:10]:
    rel, verb, ante, obj = ts
    print(f"антецедент={ante.text!r}  который={rel.text!r}  "
          f"глагол={verb.text!r}  obj={obj.text!r}")
    print(f"  → {doc[:].text[:160]}…\n")

# агрегация
from collections import Counter
pair_counter = Counter((ts[2].lemma_, ts[1].lemma_) for _, ts in hits_B)
print("Топ-15 пар (антецедент, глагол):")
for (a, v), n in pair_counter.most_common(15):
    print(f"  {a:20s} ← {v:15s} {n}")

REL_SUBJ_OBJ: 439 совпадений

антецедент='инструкций'  который='которых'  глагол='зависят'  obj='которых'
  → Таким образом , некоторые инструкции должны выполняться строго после завершения работы инструкций , от которых они зависят .…

антецедент='инструкций'  который='которых'  глагол='зависят'  obj='которых'
  → Независимые инструкции или инструкции , ставшие независимыми из-за завершения работы инструкций , от которых они зависят , могут выполняться в произвольном поря…

антецедент='набор'  который='который'  глагол='определяет'  obj='последовательность'
  → " Алгоритм - это конечный набор правил , который определяет последовательность операций для решения конкретного множества задач и обладает пятью важными чертами…

антецедент='система'  который='которая'  глагол='приводит'  obj='решению'
  → " Алгоритм - это всякая система вычислений , выполняемых по строго определённым правилам , которая после какого-либо числа шагов заведомо приводит к решению пос…

антецедент='алгоритмы'  кот

In [10]:
# Query 2: REL_SUBJ_OBJ + отглагольный антецедент (определение через действие)
from collections import Counter

def is_deverbal(noun):
    t = noun.text.lower()
    return t.endswith((
        "ание", "ания", "анию", "анием", "ании",
        "ение", "ения", "ению", "ением", "ении",
        "ация", "ации", "ацию", "ацией",
        "ство", "ства", "ству", "ством",
    ))

hits_C = [(doc, ts) for doc, ts in hits_B if is_deverbal(ts[2])]

print(f"REL_DV: {len(hits_C)} совпадений\n")
for doc, ts in hits_C[:10]:
    rel, verb, ante, obj = ts
    print(f"антецедент={ante.text!r}  который={rel.text!r}  "
          f"глагол={verb.text!r}  obj={obj.text!r}")
    print(f"  → {doc[:].text[:180]}…\n")

triplet_counter = Counter(
    (ts[2].lemma_, ts[1].lemma_, ts[3].lemma_) for _, ts in hits_C
)
print("Топ-15 троек (антецедент ← глагол → объект):")
for (a, v, o), n in triplet_counter.most_common(15):
    print(f"  {a:15s} ← {v:15s} → {o:15s} {n}")

REL_DV: 28 совпадений

антецедент='сочинение'  который='котором'  глагол='дал'  obj='котором'
  → Около 825 года он написал сочинение , в котором впервые дал описание придуманной в Индии позиционной десятичной системы счисления .…

антецедент='сочинение'  который='котором'  глагол='дал'  obj='описание'
  → Около 825 года он написал сочинение , в котором впервые дал описание придуманной в Индии позиционной десятичной системы счисления .…

антецедент='появление'  который='которому'  глагол='вошло'  obj='которому'
  → И начало ей положило появление компьютеров , благодаря которому слово " алгоритм " вошло в 1985 г. во все школьные учебники информатики и обрело новую жизнь .…

антецедент='появление'  который='которому'  глагол='вошло'  obj='учебники'
  → И начало ей положило появление компьютеров , благодаря которому слово " алгоритм " вошло в 1985 г. во все школьные учебники информатики и обрело новую жизнь .…

антецедент='преувеличением'  который='которое'  глагол='появляется'  obj='резу

## Задание 3 (4 балла)

Эссе

Прочитайте статью *FineWeb2: One Pipeline to Scale Them All — Adapting Pre-Training Data Processing to Every Language*.
Ссылка - https://arxiv.org/pdf/2506.20920

Напишите эссе, в котором вы, используя эту статью как конкретный пример, рассуждаете о том, что значит создать «хороший» многоязычный датасет для предобучения? 

Не пересказывайте статью! Выберите что-то, что вам показалось интересным (assumption, результат эксперимента, метрику и т.п.), и постройте вокруг этого аргумент. Вы можете соглашаться с авторами, спорить с ними или добавлять что-то от себя. 

Можете опираться на следующие вопросы, но можете и придумать что-то свое:

- Действительно ли возможен "one pipeline to scale them all", или некоторые аспекты отбора и очистки данных по своей природе должны быть специфичны для конкретного языка?
- Что именно означает "качество данных" в FineWeb2? Согласны ли в с этим?
- Если бы вам нужно было построить корпус для предобучения на языке, который не исследовался в статье, насколько вы бы доверяли предложенному пайплайну и где ожидали бы проблемы?
- Какое решение в дизайне FineWeb2 кажется вам наиболее убедительным — или, наоборот, наименее убедительным — и почему?
- FineWeb2 охватывает более тысячи языков, однако для многих низкоресурсных языков доступно очень мало данных, причём эти данные могут быть весьма однородными. Честно ли в этом случае заявлять об их покрытии? Как провести границу? 
- В статье сначала удаляются дубликаты, а затем часть повторяющихся данных выборочно возвращается с помощью *rehydration*. То есть дубли это не всегда "плохие данные". Как тогда понять, что дедублицировать, а что нет? 
- Авторы тщательно выбирают *early-signal* задачи для оценки, чтобы сравнивать разные варианты датасета. Достаточно ли качества модели на downstream-задачах для оценки качества корпуса?

Вы можете предлагать гипотетические примеры, альтернативные решения или эксперименты, с помощью которых можно было бы проверить ваш аргумент.

Цель задания — не покритиковать статью. Цель — построить связный аргумент, который нельзя было бы написать, не прочитав статью внимательно и не обдумав её содержание.



Мы привыкли думать, что данные - это фундамент, а пайплайн - это инструмент, который этот фундамент укладывает. Однако в статье меня привлекла не столько техническая сторона обработки, сколько вопрос: где заканчивается инструмент и начинается интерпретация? Авторы предлагают единый пайплайн для более чем тысячи языков и называют это масштабируемым решением. При этом, если проследить, как именно принимаются решения внутри этого пайплайна, становится видно, что «единый» здесь означает не «одинаково работающий», а «одинаково применяемый». Далее выясняется, что это не одно и то же.

Начну с того, что показалось мне наиболее показательным. В репозитории указано, что набор фильтров остаётся тем же, что и в оригинальном англоязычном FineWeb, а меняются только пороги. Для каждого языка отдельно настраивается confidence threshold для языкового классификатора, отключаются или ослабляются отдельные фильтры вроде `short_line_thr` и `char_dup_ratio`. Это выглядит как техническая деталь, но за ней стоит признание: английские эвристики не переносятся на другие языки напрямую, пайплайн не адаптируется сам. Его адаптируют люди, вручную, язык за языком. Для девяти языков, на которых проводились абляции, настройки подобраны осознанно. Для остальных же берётся то, что работает по умолчанию. Это не делает датасет плохим, но меняет смысл слова «масштабирование». Масштабируется здесь не качество обработки, а скорость, с которой можно заявить о покрытии нового языка.

Здесь и возникает вопрос, который статья обходит. Что значит «покрытие» языка, если данных на нём так мало, что любые статистические решения превращаются в гадание? Авторы честно пишут, что для некоторых языков доступны «very few, possibly quite homogeneous» данные. Формально датасет содержит документы на этих языках, содержательно - для сотен из них это скорее символическое присутствие. Я бы провела границу так: язык покрыт осмысленно, если модель, обученная на этих данных, показывает результаты лучше случайного угадывания на задачах, которые не сводятся к продолжению текста. Если единственное, чему модель может научиться - это предсказывать следующий токен в очень однородном корпусе, то это не предобучение, а запоминание. Авторы, кажется, это понимают, но формулируют мягче, говоря о «representation of lower resource languages», а не о решении проблемы их дефицита.

Самое сильное решение в статье, как мне кажется, это дедупликация с последующим rehydration. Авторы не удаляют дубликаты полностью: они записывают размер кластера в метаданные и возвращают часть документов обратно, с весами, зависящими от того, насколько часто документ встречался. Логика такая: если текст повторяется десять раз, это может быть важный источник, который широко цитируется. Если он повторяется тысячу раз, это, скорее всего, спам. Порог в тысячу документов отделяет одно от другого. Это разумная эвристика, но она опирается на предположение, что в корпусе достаточно разнообразия, чтобы статистика кластеров что-то значила. Для языков с сотнями тысяч документов это работает,однако для языков, где весь корпус состоит из нескольких тысяч текстов, кластер в сотню документов уже может быть заметной долей всех данных. Тогда rehydration либо усиливает то, что и так доминирует, либо не срабатывает вообще, потому что порог не достигнут. Авторы это не скрывают, но и не решают, они оставляют порог фиксированным для всех языков, хотя его смысл меняется в зависимости от объёма корпуса.

Отдельно хочется отметить оценку.В статье выбираются «early-signal» задачи - бенчмарки, на которых разница между датасетами видна уже после небольшого числа токенов. Это практичное решение: не нужно обучать модель на триллионах токенов, чтобы понять, какой корпус лучше. При этом прослеживается побочный эффект: если оценивать корпус по тому, насколько быстро модель набирает очки на узком наборе задач, можно пропустить вещи, которые проявляются только на длинной дистанции или на задачах, которых в этом наборе нет. В таблицах видно, что разные датасеты выигрывают на разных типах задач. FineWeb2 обходит CulturaX и HPLT2 на одних бенчмарках, но проигрывает на других. Это не проблема конкретно FineWeb2, это ограничение всей методологии оценки корпусов через downstream-задачи. Качество данных определяется тем, чему модель научится, а мы можем измерить только то, что умеем измерять. В статье честно показано, где результаты намекающие, а не решающие, и это, пожалуй, самое ценное в статье.

Если бы мне нужно было строить корпус для языка, которого нет в статье, я бы доверяла пайплайну частично. Языковая идентификация через GlotLID - разумный выбор, покрытие большое. Дедупликация с записью кластеров полезна даже там, где rehydration не срабатывает. Однако фильтрация мне кажется самым слабым местом. Английские фильтры, настроенные на английские пороги, могут выкидывать нормальные тексты на языках с другой структурой предложения, другой длиной слов, другим соотношением символов и пробелов. Авторы это признают, но решение (вручную настраивать пороги для каждого языка) не масштабируется. Для тысячи языков это означает тысячу наборов настроек, которые кто-то должен был сделать. В статье не сказано, сколько языков получили индивидуальную настройку, а сколько - дефолтную. Это пробел, который влияет на интерпретацию результатов сильнее, чем любая метрика.

В итоге «one pipeline to scale them all» - это скорее описание цели, чем того, как устроен FineWeb2. Пайплайн масштабируется за счёт того, что для большинства языков никакой адаптации не делается. Для тех, на которых проводились эксперименты, настройки подкручены, а для остальных - как повезёт. на мой взгляд, так работает масштабирование, когда у вас тысяча языков и ограниченное время. При этом называть это единым пайплайном для всех, значит сглаживать разницу между языками, для которых пайплайн настроен, и теми, для которых он просто применён. Возможно, самое важное здесь не то, что какой-то язык обработан хуже, а то, что сама рамка «единого решения» мешает эту разницу замечать.